In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import yaml
import os
from tqdm import tqdm

In [ ]:
from sklearn.preprocessing import scale, minmax_scale
from sklearn.metrics import root_mean_squared_error, ndcg_score
def calc_test(true_scores, pred_scores, k=10):
    rho, _ = stats.spearmanr(true_scores, pred_scores)

    # RMSE
    rmse = root_mean_squared_error(true_scores, pred_scores)

    # NDCG@k
    std_tgts = minmax_scale([true_scores], (0, 5), axis=1)
    ndcg_val = ndcg_score(std_tgts,[pred_scores], k=k)

    result ={
        'spearman': rho,
        # 'rmse': rmse,
        'ndcg': ndcg_val
    }
    return result


In [ ]:
info_df = pd.read_csv("data/info.csv")
h3_dict = info_df.set_index("PDB")["CDRH3"].to_dict()

In [ ]:
def get_ddg(path):
    results_df = pd.read_csv(path)
    ddg_scores = (results_df[results_df["scored_state"]=="ddG"]
                     .groupby("case_name")["total_score"]
                     .min()
                     .sort_index())
    return ddg_scores

def normalize_score(score):
    return (score-score.quantile(0.05))/(score.quantile(0.95)-score.quantile(0.05)+1e-10)

def ip_seq_objective(x, peak = 8, x_lower = 6.7, x_upper = 9.05):
    if x <= peak:
        return 0.5 - (0.5 / 1.3) * (x - x_lower)
    else:
        return (0.5 / (x_upper-peak)) * (x - peak)


In [ ]:
w_cols = {"acquisition_score_0": "flxddg_0_std","acquisition_score_1": "flxddg_1_std", "ablang2_perplexity": "ablang2_perplexity_std", "IP_seq": "IP_seq_std", "instability_index": "instability_index_std", "hydrophobicity": "hydrophobicity_std"}
score_cols = ["acquisition_score_0", "acquisition_score_1", "ablang2_perplexity", "IP_seq", "instability_index", "hydrophobicity"]


In [ ]:
jobdf = pd.read_csv(jobfile)
import os
cycles = {}
dfs = []
configs = []
ref_points = []
conf_datas = []
score_cols = ["acquisition_score_0", "acquisition_score_1", "ablang2_perplexity", "IP_seq", "instability_index", "hydrophobicity"]
score_cols_std = [col+"_std" for col in score_cols]

for confpath in tqdm(jobdf["CONFIG"]):
    with open(confpath) as f:
        data = yaml.safe_load(f)
    target=data["data_dir"].split("/")[1]
    model_type = data["data_dir"].split("/")[2]
    exp=data["data_dir"].split("/")[3]
    dfs_ = []
    for du_target in ["target_0","target_1"]:
        df = pd.read_csv(os.path.join(data_dir, "..", data["data_dir"], "9", du_target, "train_data", "training_data.csv"))
        df["target"]=target
        df["model_type"]=model_type
        dfs_.append(df)
    
    df = dfs_[0].copy()
    df["DMS_score_0"] = dfs_[0]["DMS_score"]
    df["DMS_score_1"] = dfs_[1]["DMS_score"]
    df = df.drop("DMS_score",axis=1)
    df["flxddg_0"] = -dfs_[0]["DMS_score"]
    df["flxddg_1"] = -dfs_[1]["DMS_score"]

    df["mutations"] = df["mutations"].fillna("")
    df["exp"]=exp

    df["acquisition_score_0"] = df["flxddg_0"]
    df["acquisition_score_1"] = df["flxddg_1"]
    df["acquisition_score_0_std"] = normalize_score(df["flxddg_0"])
    df["acquisition_score_1_std"] = normalize_score(df["flxddg_1"])
    df["ablang2_perplexity_std"] = normalize_score(df["ablang2_perplexity"])
    df["IP_seq_std"] = normalize_score(df["IP_seq"].apply(ip_seq_objective))
    df["instability_index_std"] = normalize_score(df["instability_index"])
    df["hydrophobicity_std"] = normalize_score(df["hydrophobicity"])
    
    acquisition_weight = data.get("acquisition_weight", {"acquisition_score_0": 2, "acquisition_score_1": 2})
    print(exp, acquisition_weight)
    ref_point = {}
    for col in score_cols:
        w = acquisition_weight.get(col, 0)
        df[col+"_std"] = df[col+"_std"]*w
        if w>0:
            ref_point[col+"_std"] = w
    ref_points.append(ref_point)
    df["sum_score"] = df[score_cols_std].sum(axis=1)
    
    dfs.append(df)
    configs.append({
        "target":target,
        "MAXCYCLE":10,
        "model_type": model_type,
        "exp":exp,
        "data_dir": data["data_dir"]
    })
    conf_datas.append(data)

len(dfs)

In [ ]:
test_targets = {"target_0": "4ZFG", "target_1": "4ZFF"}

In [ ]:
flex_ddg_dfs={}
sampled_seq_dfs = {}
flex_ddg_df_alls = {}
for target in test_targets.values():
    for mode in ["bias"]:
        flex_ddg_df = pd.read_csv(f"flexddg_online//flexddgs/{target}/{mode}/outputs-results.csv")
        flex_ddg_df = flex_ddg_df[flex_ddg_df["scored_state"]=="ddG"].groupby("case_name")["total_score"].min().sort_index()
        flex_ddg_dfs[target+"_"+mode]=flex_ddg_df    
        sampled_seq_dfs[target+"_"+mode]=pd.read_csv(f"flexddg_online/flexddgs/{target}/{mode}/sampled_mutations.csv", index_col=0)

test_dfs = {target: pd.read_csv(f"flexddg_online/flexddgs/{target}/bias/sampled_mutations.csv") for target in test_targets.values()}
for target in test_dfs:
    test_dfs[target]["DMS_score"] = - flex_ddg_dfs[target+"_bias"].values



In [ ]:
import yaml
from tqdm import tqdm

def hamming_distance(seq1, seq2):
    """Calculate Hamming distance between two sequences"""
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

def select_diverse_subset(df, N, score_col, ascending=False):
    """Select N diverse sequences based on score, ensuring Hamming distance > 1"""
    df_sorted = df.sort_values(score_col, ascending=ascending)
    selected_indices = []
    
    for idx in df_sorted.index:
        if len(selected_indices) >= N:
            break
            
        current_seq = df.loc[idx, "mutseq"]
        is_diverse = True
        
        # Check Hamming distance with already selected sequences
        for selected_idx in selected_indices:
            selected_seq = df.loc[selected_idx, "mutseq"]
            if hamming_distance(current_seq, selected_seq) <= 1:
                is_diverse = False
                break
        
        if is_diverse:
            selected_indices.append(idx)
    
    return df.loc[selected_indices]

In [ ]:
import yaml
from tqdm import tqdm

N=40

all_df_merges=[]
sum_df_merges=[]
for i in range(len(dfs)):
    target=configs[i]["target"]
    exp=configs[i]["exp"]
    df = dfs[i]
    CYCLE=configs[i]["MAXCYCLE"]
    
    all_dfs = {cycle+1: df[df["cycle"]<=cycle] for cycle in range(CYCLE)}
    
    cycle_dfs = {cycle+1: df[df["cycle"]==cycle] for cycle in range(CYCLE)}
    
    sum_dfs = {}
    for cycle in range(CYCLE):
        cycle_df = df[df["cycle"]<=cycle]
        sum_dfs[cycle+1] = select_diverse_subset(cycle_df, N, "sum_score", ascending=True)
    
    all_df_merge = pd.concat(all_dfs)
    sum_df_merge = pd.concat(sum_dfs)
    all_df_merge.index.names=["CYCLE", "index"]
    sum_df_merge.index.names=["CYCLE", "index"]
    all_df_merge = all_df_merge.reset_index()
    sum_df_merge = sum_df_merge.reset_index()
    all_df_merges.append(all_df_merge)
    sum_df_merges.append(sum_df_merge)
all_df_merge_cat = pd.concat(all_df_merges)
sum_df_merge_cat = pd.concat(sum_df_merges)

In [ ]:
all_test_scores_0=[]
all_test_scores_1=[]
for conf in configs:
    target = conf["target"]
    for cycle in range(10):
        input_dir_0 = os.path.join(data_dir, conf["target"], conf["model_type"], conf["exp"], str(cycle), "target_0", "train_data")
        input_dir_1 = os.path.join(data_dir, conf["target"], conf["model_type"], conf["exp"], str(cycle), "target_1", "train_data")
        test_pred_0 = np.load(os.path.join(input_dir_0, "test_inference_bias.npy"))
        test_pred_1 = np.load(os.path.join(input_dir_1, "test_inference_bias.npy"))
        test_df_0 = test_dfs[test_targets["target_0"]].copy()
        test_df_0["Pred"] = test_pred_0
        test_df_1 = test_dfs[test_targets["target_1"]].copy()
        test_df_1["Pred"] = test_pred_1
        all_test_scores_0.append({
            **calc_test(test_df_0["DMS_score"], test_df_0["Pred"]),
            "CYCLE": cycle+1,
            "target": conf["target"],
            "model_type": conf["model_type"],
            "exp": conf["exp"],
        })
        all_test_scores_1.append({
            **calc_test(test_df_1["DMS_score"], test_df_1["Pred"]),
            "CYCLE": cycle+1,
            "target": conf["target"],
            "model_type": conf["model_type"],
            "exp": conf["exp"],
        })
all_test_scores_cat_0 = pd.DataFrame(all_test_scores_0)
all_test_scores_cat_1 = pd.DataFrame(all_test_scores_1)
all_test_scores_cat_0["spearman"] = all_test_scores_cat_0["spearman"].fillna(0)
all_test_scores_cat_1["spearman"] = all_test_scores_cat_1["spearman"].fillna(0)

all_test_scores_cat_0["spearman_0"] = all_test_scores_cat_0["spearman"]
all_test_scores_cat_1["spearman_1"] = all_test_scores_cat_1["spearman"]

all_test_scores_cat_0["ndcg_0"] = all_test_scores_cat_0["ndcg"]
all_test_scores_cat_1["ndcg_1"] = all_test_scores_cat_1["ndcg"]

In [ ]:
sum_df_merge_cat.to_csv("results/flexddg_online/dual/sum_results.csv",index=False)
all_test_scores_cat_0.to_csv("results/flexddg_online/dual/all_results_test_Ang2.csv",index=False)
all_test_scores_cat_1.to_csv("results/flexddg_online/dual/all_results_test_VEGF.csv",index=False)
all_df_merge_cat.to_csv("results/flexddg_online/dual/all_results.csv",index=False)